In [ ]:
# ============================================================
# PREDICTIVE MAINTENANCE USING IoT DEVICES
# Author: Badal Raghav
# Description: Real-time sensor data simulation, ETL pipeline,
#              anomaly detection and dashboard visualisation
# ============================================================

# STEP 1 — Install & Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")

# ============================================================
# STEP 2 — GENERATE REALISTIC IoT SENSOR DATA (ETL: Extract)
# ============================================================

np.random.seed(42)
n = 500  # 500 sensor readings

# Simulate timestamps (every 10 minutes over ~3 days)
start_time = datetime(2024, 1, 1, 0, 0, 0)
timestamps = [start_time + timedelta(minutes=10*i) for i in range(n)]

# Simulate temperature with natural variation + random spikes
base_temp = 65  # Normal operating temp in °C
temperature = (
    base_temp
    + 5 * np.sin(np.linspace(0, 6 * np.pi, n))   # Natural cycle
    + np.random.normal(0, 2, n)                    # Random noise
)

# Inject realistic anomaly spikes (equipment stress events)
spike_indices = [80, 150, 230, 310, 420]
for idx in spike_indices:
    temperature[idx] += np.random.uniform(18, 28)

# Simulate vibration (mm/s) - correlated with temperature spikes
vibration = 2.5 + 0.05 * (temperature - base_temp) + np.random.normal(0, 0.3, n)

# Simulate pressure (bar)
pressure = 4.2 + 0.02 * (temperature - base_temp) + np.random.normal(0, 0.1, n)

# Create raw DataFrame
raw_df = pd.DataFrame({
    'timestamp': timestamps,
    'temperature_c': np.round(temperature, 2),
    'vibration_mms': np.round(vibration, 2),
    'pressure_bar': np.round(pressure, 2),
    'device_id': 'SENSOR_01'
})

print(f"✅ Raw sensor data generated: {len(raw_df)} readings")
print(raw_df.head())

# ============================================================
# STEP 3 — DATA CLEANING & VALIDATION (ETL: Transform)
# ============================================================

df = raw_df.copy()

# Check for missing values
missing = df.isnull().sum()
print(f"\n📋 Missing values:\n{missing}")

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f"✅ Duplicates removed: {before - len(df)}")

# Validate ranges — flag physically impossible readings
df['temp_valid'] = df['temperature_c'].between(0, 200)
df['vib_valid'] = df['vibration_mms'].between(0, 50)
df['pres_valid'] = df['pressure_bar'].between(0, 20)

valid_rows = df[df['temp_valid'] & df['vib_valid'] & df['pres_valid']]
data_quality = round(len(valid_rows) / len(df) * 100, 2)
print(f"✅ Data quality score: {data_quality}%")

df = valid_rows.drop(columns=['temp_valid', 'vib_valid', 'pres_valid'])

# ============================================================
# STEP 4 — ANOMALY DETECTION (Statistical Threshold Analysis)
# ============================================================

TEMP_THRESHOLD = 85      # °C — critical temperature limit
VIBRATION_THRESHOLD = 5  # mm/s — critical vibration limit

# Flag anomalies
df['temp_anomaly'] = df['temperature_c'] > TEMP_THRESHOLD
df['vib_anomaly'] = df['vibration_mms'] > VIBRATION_THRESHOLD
df['is_critical'] = df['temp_anomaly'] | df['vib_anomaly']

# Rolling average (smoothed trend)
df['temp_rolling_avg'] = df['temperature_c'].rolling(window=10, min_periods=1).mean()

anomaly_count = df['is_critical'].sum()
print(f"\n🚨 Critical anomalies detected: {anomaly_count}")
print(f"📊 Anomaly rate: {round(anomaly_count/len(df)*100, 2)}%")

# ============================================================
# STEP 5 — SUMMARY STATISTICS
# ============================================================

print("\n📈 Sensor Summary Statistics:")
print(df[['temperature_c', 'vibration_mms', 'pressure_bar']].describe().round(2))

# ============================================================
# STEP 6 — DASHBOARD VISUALISATION
# ============================================================

fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#0f1117')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# Color scheme
BG = '#0f1117'
CARD = '#1a1d27'
BLUE = '#4FC3F7'
RED = '#FF5252'
GREEN = '#69F0AE'
AMBER = '#FFD740'
WHITE = '#E0E0E0'
MUTED = '#888888'

def style_ax(ax, title):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines['bottom'].set_color('#2a2d3a')
    ax.spines['left'].set_color('#2a2d3a')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(title, color=WHITE, fontsize=10, fontweight='bold', pad=10)
    ax.yaxis.label.set_color(MUTED)
    ax.xaxis.label.set_color(MUTED)

x = range(len(df))
anomaly_x = df[df['is_critical']].index.tolist()
anomaly_idx = [list(df.index).index(i) for i in anomaly_x]

# --- Plot 1: Temperature over time with anomalies ---
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(x, df['temperature_c'], color=BLUE, linewidth=0.8, alpha=0.7, label='Temperature')
ax1.plot(x, df['temp_rolling_avg'], color=AMBER, linewidth=1.5, label='Rolling Avg (10 readings)')
ax1.scatter(anomaly_idx, df.loc[df['is_critical'], 'temperature_c'].values,
            color=RED, s=40, zorder=5, label='Critical Anomaly')
ax1.axhline(y=TEMP_THRESHOLD, color=RED, linestyle='--', linewidth=1, alpha=0.6, label=f'Threshold ({TEMP_THRESHOLD}°C)')
ax1.fill_between(x, df['temperature_c'], alpha=0.08, color=BLUE)
style_ax(ax1, '🌡️  Temperature Monitoring — Live Sensor Feed')
ax1.set_ylabel('Temperature (°C)')
ax1.set_xlabel('Reading Index')
ax1.legend(fontsize=8, facecolor=CARD, edgecolor='#2a2d3a', labelcolor=WHITE, loc='upper right')

# --- Plot 2: Vibration ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(x, df['vibration_mms'], color=GREEN, linewidth=0.8, alpha=0.8)
ax2.axhline(y=VIBRATION_THRESHOLD, color=RED, linestyle='--', linewidth=1, alpha=0.7, label=f'Threshold ({VIBRATION_THRESHOLD} mm/s)')
ax2.fill_between(x, df['vibration_mms'], alpha=0.1, color=GREEN)
style_ax(ax2, '📳  Vibration Levels (mm/s)')
ax2.set_ylabel('Vibration (mm/s)')
ax2.legend(fontsize=7, facecolor=CARD, edgecolor='#2a2d3a', labelcolor=WHITE)

# --- Plot 3: Pressure ---
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(x, df['pressure_bar'], color=AMBER, linewidth=0.8, alpha=0.8)
ax3.fill_between(x, df['pressure_bar'], alpha=0.1, color=AMBER)
style_ax(ax3, '🔵  Pressure Levels (bar)')
ax3.set_ylabel('Pressure (bar)')

# --- Plot 4: Anomaly Distribution ---
ax4 = fig.add_subplot(gs[2, 0])
labels = ['Normal', 'Critical']
sizes = [len(df) - anomaly_count, anomaly_count]
colors = [GREEN, RED]
wedges, texts, autotexts = ax4.pie(sizes, labels=labels, colors=colors,
                                    autopct='%1.1f%%', startangle=90,
                                    textprops={'color': WHITE, 'fontsize': 9})
for at in autotexts:
    at.set_color(BG)
    at.set_fontweight('bold')
ax4.set_facecolor(CARD)
ax4.set_title('🚨  Anomaly Distribution', color=WHITE, fontsize=10, fontweight='bold', pad=10)

# --- Plot 5: Temperature Histogram ---
ax5 = fig.add_subplot(gs[2, 1])
ax5.hist(df['temperature_c'], bins=30, color=BLUE, alpha=0.8, edgecolor=CARD)
ax5.axvline(x=TEMP_THRESHOLD, color=RED, linestyle='--', linewidth=1.5, label=f'Critical threshold')
ax5.axvline(x=df['temperature_c'].mean(), color=AMBER, linestyle='--', linewidth=1.5, label=f'Mean: {df["temperature_c"].mean():.1f}°C')
style_ax(ax5, '📊  Temperature Distribution')
ax5.set_ylabel('Frequency')
ax5.set_xlabel('Temperature (°C)')
ax5.legend(fontsize=7, facecolor=CARD, edgecolor='#2a2d3a', labelcolor=WHITE)

# Title
fig.suptitle('PREDICTIVE MAINTENANCE DASHBOARD — IoT Sensor Analytics',
             color=WHITE, fontsize=14, fontweight='bold', y=0.98)

plt.savefig('predictive_maintenance_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.show()
print("\n✅ Dashboard saved as 'predictive_maintenance_dashboard.png'")

# ============================================================
# STEP 7 — EXPORT CLEANED DATA
# ============================================================

df.to_csv('cleaned_sensor_data.csv', index=False)
print("✅ Cleaned data exported as 'cleaned_sensor_data.csv'")

# ============================================================
# STEP 8 — FINAL REPORT
# ============================================================

print("\n" + "="*55)
print("         PREDICTIVE MAINTENANCE — FINAL REPORT")
print("="*55)
print(f"  Total Readings Analysed  : {len(df)}")
print(f"  Data Quality Score       : {data_quality}%")
print(f"  Avg Temperature          : {df['temperature_c'].mean():.2f} °C")
print(f"  Max Temperature          : {df['temperature_c'].max():.2f} °C")
print(f"  Avg Vibration            : {df['vibration_mms'].mean():.2f} mm/s")
print(f"  Critical Anomalies       : {anomaly_count}")
print(f"  Anomaly Rate             : {round(anomaly_count/len(df)*100,2)}%")
print(f"  Recommendation           : {'⚠️  Schedule maintenance' if anomaly_count > 10 else '✅ System operating normally'}")
print("="*55)